# Activity recognition · Neighborhood shares and window summaries
Compare the previous SMOTE–Tomek and balanced-weight boosters on the same four grouped folds. No participant, session, timestamp or true activity label is required for an incoming observation.

For each reading, calculate a distance-weighted distribution of labels among 15 distinct training readings. Test standardized Euclidean distance and a supervised shrinkage-LDA projection separately. Training shares are cross-fitted by duplicate group in three inner folds. Each outer validation query uses only real outer-training reference rows. The six window summaries describe standardized anonymized columns. This is a development comparison across previously examined folds.

In [1]:
from pathlib import Path
from collections import deque
import zipfile
import shutil
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 610
import os
WORKERS = min(8, os.cpu_count() or 1)
print("scikit-learn", sklearn.__version__)
def locate(name):
    for root in (Path.cwd(), Path.cwd().parent):
        direct = root / name
        if direct.is_file():
            return direct
        data = root / "Docs and Data"
        if data.is_dir():
            match = next(data.rglob(name), None)
            if match:
                return match

har_path=locate("har_train.csv")
if not har_path:
    from google.colab import files
    files.upload()
    har_path=locate("har_train.csv")
har=pd.read_csv(har_path,header=None)


scikit-learn 1.9.1


In [2]:
import importlib.util
import subprocess
import sys
if any(importlib.util.find_spec(k) is None for k in ("imblearn","flaml","xgboost","lightgbm")):
    subprocess.check_call([sys.executable,"-m","pip","install","imbalanced-learn==0.14.2",
                           "flaml==2.8.0","xgboost","lightgbm"])
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from flaml.automl.model import XGBoostSklearnEstimator, LGBMEstimator
from collections import Counter

def booster(family, config, task):
    wrapper = {"xgboost":XGBoostSklearnEstimator, "lgbm":LGBMEstimator}[family](
        task=task,n_jobs=WORKERS,**config)
    return wrapper.estimator_class(**wrapper.params)

def resample(matrix, y, strategy, k):
    sampler = SMOTETomek(smote=SMOTE(sampling_strategy=strategy,k_neighbors=k,random_state=SEED),
                         tomek=TomekLinks(sampling_strategy="all",n_jobs=1),random_state=SEED)
    return sampler.fit_resample(matrix,y)

activity_configs=[('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523}), ('lgbm', {'n_estimators': 100, 'num_leaves': 4, 'min_child_samples': 16, 'learning_rate': 0.07832503615697724, 'log_max_bin': 9, 'colsample_bytree': 0.8814460015050225, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.024037654653852953}), ('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523}), ('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523})]


In [3]:
Xh, yh = har.iloc[:,:-1], har.iloc[:,-1].to_numpy()
groups = pd.util.hash_pandas_object(Xh, index=False).to_numpy()
labels = np.unique(yh)
# Accept a split only if all activities occur on both sides of every fold.
for split_seed in range(SEED, SEED+200):
    splitter = StratifiedGroupKFold(4, shuffle=True, random_state=split_seed)
    har_folds = list(splitter.split(Xh, yh, groups))
    if all(len(np.unique(yh[tr])) == len(labels) == len(np.unique(yh[va])) for tr,va in har_folds):
        break
else:
    raise ValueError("No four-fold split covers all activities.")
print("Fold seed:", split_seed)
labels = np.unique(yh)
ycode = np.searchsorted(labels,yh)


Fold seed: 610


In [4]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_sample_weight
def row_summaries(z):
    return np.column_stack([np.mean(abs(z),axis=1),np.sqrt(np.mean(z*z,axis=1)),
                            np.mean(abs(z)>2,axis=1),np.quantile(z,[.1,.5,.9],axis=1).T])

def shares(ref,query,ids,group_ids,project):
    scale=StandardScaler().fit(ref)
    keep=scale.var_>0
    a=scale.transform(ref)[:,keep]
    b=scale.transform(query)[:,keep]
    if project:
        lda=LinearDiscriminantAnalysis(solver="eigen",shrinkage="auto").fit(a,ids)
        a,b=lda.transform(a),lda.transform(b)
    # Equal contribution per distinct reading, even if it appears many times.
    unique,first,inv=np.unique(group_ids,return_index=True,return_inverse=True)
    votes=np.zeros((len(unique),len(labels)))
    np.add.at(votes,(inv,ids),1)
    votes/=votes.sum(axis=1,keepdims=True)
    distance,near=NearestNeighbors(n_neighbors=15).fit(a[first]).kneighbors(b)
    weight=1/np.maximum(distance,1e-8)
    return np.sum(votes[near]*weight[:,:,None],axis=1)/weight.sum(axis=1,keepdims=True)

variants=["Incumbent","Window summaries","Raw-space shares","Activity-space shares","All features"]
policies=["SMOTE-Tomek","Balanced weights"]
predictions={(v,p):np.empty_like(ycode) for v in variants for p in policies}
fold_ids=np.zeros(len(yh),int)
for fold,(tr,va) in enumerate(har_folds):
    fold_ids[va]=fold+1
    assert not np.intersect1d(groups[tr],groups[va]).size
    x,v=Xh.iloc[tr].to_numpy(),Xh.iloc[va].to_numpy()
    y=ycode[tr]; scale=StandardScaler().fit(x)
    xs,vs=scale.transform(x),scale.transform(v)
    rows=(row_summaries(xs[:,scale.var_>0]),row_summaries(vs[:,scale.var_>0]))
    counts=Counter(y);goal=int(np.median(list(counts.values())))
    strategy={c:goal for c,n in counts.items() if n<goal}
    for seed in range(SEED,SEED+200):
        inner=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(x,y,groups[tr]))
        if all(len(np.unique(y[a]))==len(labels) for a,b in inner): break
    else: raise ValueError("Missing a reference class")
    features={}
    for key,project in [("Raw-space shares",False),("Activity-space shares",True)]:
        out=np.empty((len(y),len(labels)))
        for a,b in inner:
            assert not np.intersect1d(groups[tr[a]],groups[tr[b]]).size
            out[b]=shares(x[a],x[b],y[a],groups[tr[a]],project)
        features[key]=(out,shares(x,v,y,groups[tr],project))
    features["Window summaries"]=rows
    features["All features"]=tuple(np.column_stack([pair[i] for pair in features.values()]) for i in (0,1))
    fam,cfg=activity_configs[fold]
    for name in variants:
        xx,vv=xs,vs
        if name!="Incumbent":
            f,g=features[name]
            norm=StandardScaler().fit(f)
            xx,vv=np.column_stack((xs,norm.transform(f))),np.column_stack((vs,norm.transform(g)))
        xr,yr=resample(xx,y,strategy,min(3,min(counts.values())-1))
        for policy in policies:
            model=booster(fam,cfg,"multiclass")
            if policy=="SMOTE-Tomek":model.fit(xr,yr)
            else:model.fit(xx,y,sample_weight=compute_sample_weight("balanced",y))
            predictions[name,policy][va]=model.predict(vv)
    print("Completed grouped fold",fold+1,flush=True)

scores=[]
for (v,p),pr in predictions.items():
    base=predictions["Incumbent",p]
    delta=[int(((pr==ycode)&(fold_ids==k)).sum()-((base==ycode)&(fold_ids==k)).sum()) for k in range(1,5)]
    scores.append(dict(features=v,training=p,correct=int((pr==ycode).sum()),accuracy=accuracy_score(ycode,pr),
                       macro_f1=f1_score(ycode,pr,average="macro"),net_by_fold=str(delta)))
board=pd.DataFrame(scores).sort_values(["correct","macro_f1"],ascending=False)
display(board)
best=board.iloc[0]
a=predictions["Incumbent","SMOTE-Tomek"];b=predictions[best.features,best.training]
display(pd.DataFrame([dict(activity=k,support=int((yh==k).sum()),baseline_correct=int(((a==ycode)&(yh==k)).sum()),
                           winner_correct=int(((b==ycode)&(yh==k)).sum())) for k in labels]))

Completed grouped fold 1
Completed grouped fold 2
Completed grouped fold 3
Completed grouped fold 4
             features         training  correct  accuracy  macro_f1       net_by_fold
     Raw-space shares Balanced weights     1957  0.801392  0.705833    [8, 7, 12, 10]
            Incumbent      SMOTE-Tomek     1950  0.798526  0.668414      [0, 0, 0, 0]
     Window summaries      SMOTE-Tomek     1946  0.796888  0.662869   [3, 5, -10, -2]
     Window summaries Balanced weights     1942  0.795250  0.691108     [4, 2, 16, 0]
Activity-space shares Balanced weights     1937  0.793202  0.691946   [-1, 8, 11, -1]
         All features      SMOTE-Tomek     1936  0.792793  0.653056 [-4, 10, -9, -11]
         All features Balanced weights     1925  0.788288  0.663647     [-5, 2, 6, 2]
Activity-space shares      SMOTE-Tomek     1923  0.787469  0.658749 [-12, 8, -14, -9]
     Raw-space shares      SMOTE-Tomek     1921  0.786650  0.641736 [-6, -1, -20, -2]
            Incumbent Balanced weights  